# Step 6: Split prescriptions into therapies

In [ ]:
import pyspark
import re
import dxpy
from scipy import stats
import hail as hl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import sys
import importlib
sys.path.append('../')
import prescriptions_processing

In [ ]:
importlib.reload(prescriptions_processing)

from prescriptions_processing import TherapiesSplitter

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)
hl.init(sc=sc, default_reference='GRCh38')

### Environment setup check

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Configuration and Hail tables loading

In [ ]:
input_database = 'prescriptions_db'
input_tb = 'cleaned_prescriptions_with_doses_v6.2.0.ht'

second_output_database = 'prescriptions_db'
second_output_tb = 'cleaned_prescriptions_splited_to_therapies_v6.2.0.ht'

project_id = dxpy.PROJECT_CONTEXT_ID

In [ ]:
input_db_id = dxpy.find_one_data_object(name=input_database, classname='database', project=project_id)['id']
ht = hl.read_table(f'dnax://{input_db_id}/{input_tb}')

In [ ]:
splitter = TherapiesSplitter(
            ht=ht,
            time_gap_days=60,
            use_statistical_dose_check=True,
            sd_fraction=2.0,
            window_size=5,
            batch_size=10000)

In [ ]:
result_ht = splitter.split()

In [ ]:
result_ht.describe()

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {second_output_database} LOCATION 'dnax://'")
second_output_db_id = dxpy.find_one_data_object(name=second_output_database, classname='database', project=project_id)['id']
second_output_url = f'dnax://{second_output_db_id}/{second_output_tb}'

%time result_ht.write(second_output_url, overwrite=True)